# Hydranomic - Pipeline Analítico WEGnology Notebook

Este Jupyter Notebook foi desenvolvido e adaptado especificamente para a metodologia de execução de **Notebooks na Plataforma IoT WEGnology** (baseada na arquitetura Losant).

### Como funciona o fluxo de dados na WEGnology:
1. **Entradas (Inputs Tab):** O container do WEGnology injeta arquivos na pasta indicada por `os.environ['INPUT_DIR']` (ex: `conta_povoada.json`, `config.json` ou `context.json` via Workflow).
2. **Processamento:** Tratamento de dados de consumo, treino de modelo de Machine Learning (`RandomForestRegressor`), projeção de consumo/fatura e cálculo do gráfico anual.
3. **Saídas (Outputs Tab):** Todos os arquivos gravados na pasta `os.environ['OUTPUT_DIR']` são automaticamente sincronizados com os **Application Files** (armazenamento persistente de arquivos da sua aplicação WEGnology) ou inseridos em **Data Tables**.

---
### Artefatos Gerados:
- `grafico_anual.json`: Histórico e distribuição mensal do consumo anual (em m³).
- `previsao_ml.json`: Previsão de consumo para o próximo ciclo e valor estimado da fatura.
- `progresso_meta.json`: Acompanhamento de metas ativas de consumo.

In [ ]:
# 1. Configuração do Ambiente e Importação de Módulos
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Importa os adaptadores WEGnology desenvolvidos para o Hydranomic
from wegnology_adapter import (
    obter_diretorio_input,
    obter_diretorio_output,
    carregar_json,
    salvar_json,
    carregar_contexto
)

print(f"[WEGnology] Diretório de Entrada (INPUT_DIR): {obter_diretorio_input()}")
print(f"[WEGnology] Diretório de Saída (OUTPUT_DIR): {obter_diretorio_output()}")

In [ ]:
# 2. Célula de Parâmetros (Injetáveis via Workflow 'Notebook: Execute' do WEGnology)
# Se executado por um workflow, esses parâmetros podem ser sobrescritos dinamicamente
ANO_REFERENCIA = None
TARIFA_M3 = 59.6
ARQUIVO_ENTRADA = "conta_povoada.json"
SALVAR_ARQUIVOS = True

## 3. Leitura e Preparação dos Dados de Entrada

In [ ]:
# Carrega dados a partir do INPUT_DIR configurado na WEGnology
dados_brutos = carregar_json(ARQUIVO_ENTRADA)
print(f"Total de registros carregados: {len(dados_brutos)}")

df_consumo = pd.DataFrame(dados_brutos)
df_consumo.head()

## 4. Geração do Dataset para Gráfico Anual de Consumo (`grafico_anual.json`)
Gera o payload estruturado para alimentar o painel ou Experience Page do Hydranomic.

In [ ]:
from data_chart import processar_dados_grafico

# Processa os dados de consumo para o ano de referência selecionado
grafico_resultado = processar_dados_grafico(
    dados=dados_brutos,
    ano_referencia=ANO_REFERENCIA,
    salvar=SALVAR_ARQUIVOS
)

print(f"Ano Selecionado: {grafico_resultado['ano_referencia']}")
print(f"Mês Atual: {grafico_resultado['mes_atual']}")

# Visualização rápida no Notebook
meses_nomes = [m['rotulo'] for m in grafico_resultado['meses']]
consumos = [m['consumo_m3'] if m['consumo_m3'] is not None else 0 for m in grafico_resultado['meses']]

plt.figure(figsize=(10, 4))
plt.bar(meses_nomes, consumos, color='#0284c7')
plt.title(f"Consumo Mensal de Água ({grafico_resultado['ano_referencia']}) - m³")
plt.xlabel("Mês")
plt.ylabel("Consumo (m³)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Treinamento do Modelo de Machine Learning (`RandomForestRegressor`)
Treina o modelo regressivo considerando a sazonalidade do mês, o tempo linear e a defasagem (lag de consumo do mês anterior).

In [ ]:
from machinelearning import preparar_dados, treinar_modelo, prever_proximo_ciclo

# 1. Engenharia de Features
df_completo, df_treino, features_entrada, variavel_saida = preparar_dados(df_consumo)

# 2. Treinamento
modelo_rf = treinar_modelo(df_treino, features_entrada, variavel_saida)
print(f"Modelo treinado com sucesso! Features utilizadas: {features_entrada}")

# Importância das Features
for feature, imp in zip(features_entrada, modelo_rf.feature_importances_):
    print(f" - Feature '{feature}': importância = {imp:.4f}")

## 6. Previsão de Consumo e Projeção Financeira (`previsao_ml.json`)
Projeta o consumo esperado para a próxima fatura e calcula o custo estimado em Reais.

In [ ]:
previsao_resultado = prever_proximo_ciclo(
    modelo=modelo_rf,
    df_completo=df_completo,
    features_entrada=features_entrada,
    tarifa=TARIFA_M3
)

if SALVAR_ARQUIVOS:
    salvar_json("previsao_ml.json", previsao_resultado)

print("=== PREVISÃO PARA O PRÓXIMO CICLO ===")
print(f"Período Previsto: {previsao_resultado['mes']:02d}/{previsao_resultado['ano']}")
print(f"Unidade Consumidora: {previsao_resultado['unidade_consumidora']}")
print(f"Consumo Previsto: {previsao_resultado['previsao_m3']} m³")
print(f"Tarifa Aplicada: R$ {previsao_resultado['tarifa_m3']} por m³")
print(f"Valor Estimado da Fatura: R$ {previsao_resultado['valor_estimado_conta']:.2f}")

## 7. Acompanhamento de Metas de Consumo (`progresso_meta.json`)
Avalia se a meta ativa foi ultrapassada ou se está na faixa de alerta (80%).

In [ ]:
from metas import avaliar_progresso_meta, carregar_config

# Obtém o último consumo registrado
ultimo_consumo = previsao_resultado["ultimo_consumo_registrado_m3"]

relatorio_meta = avaliar_progresso_meta(ultimo_consumo)
config_atual = carregar_config()

if relatorio_meta:
    print("=== SITUAÇÃO DA META ATIVA ===")
    print(f"Meta: {relatorio_meta['meta_nome']}")
    print(f"Limite: {relatorio_meta['limite_m3']} m³ | Consumo Atual: {relatorio_meta['consumo_atual_m3']} m³")
    print(f"Progresso: {relatorio_meta['percentual_atingido']}% (Situação: {relatorio_meta['situacao']})")
else:
    print("Nenhuma meta de consumo ativa configurada no momento.")

## 8. Sincronização e Destinos de Salvamento na WEGnology

Todos os arquivos salvos em `OUTPUT_DIR` estão disponíveis para serem mapeados na aba **Outputs** do Notebook no portal WEGnology:
- `OUTPUT_DIR/grafico_anual.json` -> Salvo em **Application Files**: `/workflows/grafico_anual.json`
- `OUTPUT_DIR/previsao_ml.json` -> Salvo em **Application Files**: `/workflows/previsao_ml.json`
- `OUTPUT_DIR/progresso_meta.json` -> Salvo em **Application Files**: `/workflows/progresso_meta.json`

In [ ]:
dir_saida = obter_diretorio_output()
print("Arquivos disponíveis no OUTPUT_DIR para sincronização com o WEGnology:")
if dir_saida.exists():
    for arq in dir_saida.glob("*.json"):
        tamanho = arq.stat().st_size
        print(f" [✓] {arq.name} ({tamanho} bytes) -> Pronto para persistência na WEGnology")